<a href="https://colab.research.google.com/github/josemesa0112/machine-learning-repository/blob/main/Machine_Learning_applied_to_Crime_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INSTALACIÓN Y CARGA DEL DATASET**

In [1]:
!pip install ucimlrepo -q

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
communities = fetch_ucirepo(id=183)

X = communities.data.features
y = communities.data.targets

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)
print("Variable objetivo:", y.columns.tolist())

Dimensión de X: (1994, 127)
Dimensión de y: (1994, 1)
Variable objetivo: ['ViolentCrimesPerPop']


## **UNIFICAR DATOS**

In [5]:
df = pd.concat([X, y], axis=1)

print("Dimensión completa del dataset:", df.shape)
df.head()

Dimensión completa del dataset: (1994, 128)


,state,county,community,communityname,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,...,LandArea,PopDens,PctUsePubTrans,PolicCars,PolicOperBudg,LemasPctPolicOnPatr,LemasGangUnitDeploy,LemasPctOfficDrugUn,PolicBudgPerPop,ViolentCrimesPerPop
0,8,?,?,Lakewoodcity,1,0.19,0.33,0.02,0.90,0.12,...,0.12,0.26,0.20,0.06,0.04,0.9,0.5,0.32,0.14,0.20
1,53,?,?,Tukwilacity,1,0.00,0.16,0.12,0.74,0.45,...,0.02,0.12,0.45,?,?,?,?,0.00,?,0.67
2,24,?,?,Aberdeentown,1,0.00,0.42,0.49,0.56,0.17,...,0.01,0.21,0.02,?,?,?,?,0.00,?,0.43
3,34,5,81440,Willingborotownship,1,0.04,0.77,1.00,0.08,0.12,...,0.02,0.39,0.28,?,?,?,?,0.00,?,0.12
4,42,95,6096,Bethlehemtownship,1,0.01,0.55,0.02,0.95,0.09,...,0.04,0.09,0.02,?,?,?,?,0.00,?,0.03


## **REVISIÓN DE ESTRUCTURA GENERAL**

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Columns: 128 entries, state to ViolentCrimesPerPop
dtypes: float64(100), int64(2), object(26)
memory usage: 1.9+ MB


In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
state,1994.0,28.683551,16.397553,1.0,12.00,34.00,42.00,56.0
fold,1994.0,5.493982,2.873694,1.0,3.00,5.00,8.00,10.0
population,1994.0,0.057593,0.126906,0.0,0.01,0.02,0.05,1.0
householdsize,1994.0,0.463395,0.163717,0.0,0.35,0.44,0.54,1.0
racepctblack,1994.0,0.179629,0.253442,0.0,0.02,0.06,0.23,1.0
...,...,...,...,...,...,...,...,...
LandArea,1994.0,0.065231,0.109459,0.0,0.02,0.04,0.07,1.0
PopDens,1994.0,0.232854,0.203092,0.0,0.10,0.17,0.28,1.0
PctUsePubTrans,1994.0,0.161685,0.229055,0.0,0.02,0.07,0.19,1.0
LemasPctOfficDrugUn,1994.0,0.094052,0.240328,0.0,0.00,0.00,0.00,1.0


## **DEFINIR VARIABLE OBJETIVO**

In [8]:
target_col = "ViolentCrimesPerPop"

print("Variable objetivo:", target_col)
print(df[target_col].describe())

Variable objetivo: ViolentCrimesPerPop
count    1994.000000
mean        0.237979
std         0.232985
min         0.000000
25%         0.070000
50%         0.150000
75%         0.330000
max         1.000000
Name: ViolentCrimesPerPop, dtype: float64


## **ELIMINAR COLUMNAS IDENTIFICADORAS**

Estas columnas no aportan información relevante para la predicción ya que solo son identificadores.

In [9]:
cols_to_drop = []

for col in ["state", "county", "community", "communityname", "fold"]:
    if col in df.columns:
        cols_to_drop.append(col)

print("Columnas eliminadas:", cols_to_drop)

df_model = df.drop(columns=cols_to_drop)

print("Dimensión después de eliminar identificadores:", df_model.shape)

Columnas eliminadas: ['state', 'county', 'community', 'communityname', 'fold']
Dimensión después de eliminar identificadores: (1994, 123)


In [20]:
#print(df_model.dtypes[df_model.dtypes != "float64"])
print(" ")
print(df_model.isna().sum()[df_model.isna().sum() == 0])

 
population             0
householdsize          0
racepctblack           0
racePctWhite           0
racePctAsian           0
                      ..
LemasPctPolicOnPatr    0
LemasGangUnitDeploy    0
LemasPctOfficDrugUn    0
PolicBudgPerPop        0
ViolentCrimesPerPop    0
Length: 123, dtype: int64


## **CONVERTIR VARIABLES A NUMÉRICAS**

In [23]:
df_model= df_model.apply(pd.to_numeric, errors="coerce")

print(df_model.dtypes.value_counts())

float64    123
Name: count, dtype: int64


## **REVISAR DATOS FALTANTES**

In [24]:
missing_values = df_model.isnull().sum()
missing_percent = (missing_values / len(df_model)) * 100

missing_table = pd.DataFrame({
    "missing_values": missing_values,
    "missing_percent": missing_percent
}).sort_values(by="missing_percent", ascending=False)

missing_table[missing_table["missing_values"] > 0].head(30)

,missing_values,missing_percent
PolicAveOTWorked,1675,84.002006
LemasTotalReq,1675,84.002006
LemasSwFTFieldPerPop,1675,84.002006
PctPolicWhite,1675,84.002006
RacialMatchCommPol,1675,84.002006
LemasSwFTPerPop,1675,84.002006
LemasSwFTFieldOps,1675,84.002006
PolicReqPerOffic,1675,84.002006
LemasTotReqPerPop,1675,84.002006
LemasSwornFT,1675,84.002006


## **IMPUTACIÓN DE DATOS FALTANTES**

In [33]:
svmImputer = SVR(kernel='rbf')

svmImputer = IterativeImputer(estimator = svmImputer, max_iter=10, random_state=42)

df_model_imputed = pd.DataFrame(svmImputer.fit_transform(df_model), columns=df_model.columns)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [41]:
df_model_imputed.isna().sum()[df_model_imputed.isna().sum() != 0] #verificando los datos faltantes luego de la imputación

,0


## **ELIMINAR COLUMNAS CON MUCHOS DATOS FALTANTES**

Columnas con más del 40% de valores faltantes.

In [42]:
threshold_missing = 40

cols_high_missing = missing_table[
    missing_table["missing_percent"] > threshold_missing
].index.tolist()

# Nunca eliminar la variable objetivo
if target_col in cols_high_missing:
    cols_high_missing.remove(target_col)

print("Columnas con más del 40% de datos faltantes:")
print(cols_high_missing)

df_model = df_model.drop(columns=cols_high_missing)

print("Dimensión después de eliminar columnas con muchos faltantes:", df_model.shape)

Columnas con más del 40% de datos faltantes:
['PolicAveOTWorked', 'LemasTotalReq', 'LemasSwFTFieldPerPop', 'PctPolicWhite', 'RacialMatchCommPol', 'LemasSwFTPerPop', 'LemasSwFTFieldOps', 'PolicReqPerOffic', 'LemasTotReqPerPop', 'LemasSwornFT', 'PolicPerPop', 'PolicBudgPerPop', 'LemasGangUnitDeploy', 'LemasPctPolicOnPatr', 'PolicCars', 'PolicOperBudg', 'PctPolicMinor', 'PctPolicAsian', 'PctPolicHisp', 'OfficAssgnDrugUnits', 'NumKindsDrugsSeiz', 'PctPolicBlack']
Dimensión después de eliminar columnas con muchos faltantes: (1994, 101)


Para la variable 'OtherPerCap' como solo tiene un dato faltante, le imputamos la media

In [55]:
df_model['OtherPerCap'][df_model['OtherPerCap'].isna() != 0] = df_model['OtherPerCap'].mean()

/tmp/ipykernel_538/720923645.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_model['OtherPerCap'][df_model['OtherPerCap'].isna() != 0] = df_model['OtherPerCap'].mean()


In [56]:
df_model.isna().sum()[df_model.isna().sum() != 0] #verificando valores faltantes luego

,0


**VARIABLES CON BAJA VARIABILIDAD**

Una variable casi constante aporta poca información.

In [57]:
X_temp = df_model.drop(columns=[target_col])

low_variance_cols = []

for col in X_temp.columns:
    if X_temp[col].nunique(dropna=True) <= 1:
        low_variance_cols.append(col)

print("Columnas con baja variabilidad:", low_variance_cols)

df_model = df_model.drop(columns=low_variance_cols)

print("Dimensión después de eliminar baja variabilidad:", df_model.shape)

Columnas con baja variabilidad: []
Dimensión después de eliminar baja variabilidad: (1994, 101)


Para el dataset con imputación de datos

In [58]:
X_temp = df_model_imputed.drop(columns=[target_col])

low_variance_cols = []

for col in X_temp.columns:
    if X_temp[col].nunique(dropna=True) <= 1:
        low_variance_cols.append(col)

print("Columnas con baja variabilidad:", low_variance_cols)

df_model_imputed = df_model_imputed.drop(columns=low_variance_cols)

print("Dimensión después de eliminar baja variabilidad:", df_model_imputed.shape)

Columnas con baja variabilidad: []
Dimensión después de eliminar baja variabilidad: (1994, 123)


**VARIABLES MUY CORRELACIONADAS ENTRE SÍ**

Se eliminan variables correlacionadas entre si para reducir la redundancia

In [65]:
X_corr = df_model.drop(columns=[target_col]).corr().abs()

upper = X_corr.where(
    np.triu(np.ones(X_corr.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper.columns
    if any(upper[column] > 0.90)
]

print("Cantidad de variables altamente correlacionadas entre sí:", len(high_corr_features))
high_corr_features

Cantidad de variables altamente correlacionadas entre sí: 32


['agePct16t24',
 'numbUrban',
 'pctWSocSec',
 'medFamInc',
 'perCapInc',
 'whitePerCap',
 'NumUnderPov',
 'PctNotHSGrad',
 'PctOccupMgmtProf',
 'FemalePctDiv',
 'TotalPctDiv',
 'PctKids2Par',
 'PctYoungKids2Par',
 'PctTeen2Par',
 'NumIlleg',
 'PctImmigRec5',
 'PctImmigRec8',
 'PctImmigRec10',
 'PctRecImmig5',
 'PctRecImmig8',
 'PctRecImmig10',
 'PctNotSpeakEnglWell',
 'PctLargHouseOccup',
 'PersPerOccupHous',
 'PersPerOwnOccHous',
 'PctHousOwnOcc',
 'OwnOccMedVal',
 'OwnOccHiQuart',
 'RentMedian',
 'RentHighQ',
 'MedRent',
 'PctForeignBorn']

Luego con el dataset con imputación de datos

In [66]:
X_corr = df_model_imputed.drop(columns=[target_col]).corr().abs()

upper = X_corr.where(
    np.triu(np.ones(X_corr.shape), k=1).astype(bool)
)

high_corr_features_imputed = [
    column for column in upper.columns
    if any(upper[column] > 0.90)
]

print("Cantidad de variables altamente correlacionadas entre sí:", len(high_corr_features_imputed))
high_corr_features_imputed

Cantidad de variables altamente correlacionadas entre sí: 38


['agePct16t24',
 'numbUrban',
 'pctWSocSec',
 'medFamInc',
 'perCapInc',
 'whitePerCap',
 'NumUnderPov',
 'PctNotHSGrad',
 'PctOccupMgmtProf',
 'FemalePctDiv',
 'TotalPctDiv',
 'PctKids2Par',
 'PctYoungKids2Par',
 'PctTeen2Par',
 'NumIlleg',
 'PctImmigRec5',
 'PctImmigRec8',
 'PctImmigRec10',
 'PctRecImmig5',
 'PctRecImmig8',
 'PctRecImmig10',
 'PctNotSpeakEnglWell',
 'PctLargHouseOccup',
 'PersPerOccupHous',
 'PersPerOwnOccHous',
 'PctHousOwnOcc',
 'OwnOccMedVal',
 'OwnOccHiQuart',
 'RentMedian',
 'RentHighQ',
 'MedRent',
 'PctForeignBorn',
 'LemasSwFTFieldOps',
 'LemasSwFTFieldPerPop',
 'PolicPerPop',
 'OfficAssgnDrugUnits',
 'PolicOperBudg',
 'PolicBudgPerPop']

**VERSIÓN REDUCIDA DEL DATASET**

Eliminamos variables redundantes con correlación mayor a 0.90.

In [67]:
df_reduced = df_model.drop(columns=high_corr_features)

print("Dimensión original después de limpieza básica:", df_model.shape)
print("Dimensión después de eliminar variables redundantes:", df_reduced.shape)

Dimensión original después de limpieza básica: (1994, 101)
Dimensión después de eliminar variables redundantes: (1994, 69)


Con el dataset de datos imputados

In [69]:
df_reduced_imputed = df_model_imputed.drop(columns=high_corr_features_imputed)

print("Dimensión original después de limpieza básica:", df_model_imputed.shape)
print("Dimensión después de eliminar variables redundantes:", df_reduced_imputed.shape)

Dimensión original después de limpieza básica: (1994, 123)
Dimensión después de eliminar variables redundantes: (1994, 85)


**SEPARAR VARIABLES PREDICTORAS Y OBJETIVO**

In [71]:
X_final = df_reduced.drop(columns=[target_col])
X_final_imputed = df_reduced_imputed.drop(columns=[target_col]) #para el dataset de datos imputados
y_final = df_reduced[target_col]

print("X_final:", X_final.shape)
print("X_final_imputed:", X_final_imputed.shape)
print("y_final:", y_final.shape)

X_final: (1994, 68)
X_final_imputed: (1994, 84)
y_final: (1994,)


**15) DIVISIÓN ENTRENAMIENTO-TEST**

80% para entrenamiento-validación y 20% para test final.

In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.20,
    random_state=42
)

X_train_imputed, X_test_imputed, y_train_imputed, y_test_imputed = train_test_split( # para los datos imputados
    X_final_imputed,
    y_final,
    test_size=0.20,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Train_imputed:", X_train_imputed.shape)
print("Test_imputed:", X_test_imputed.shape)

Train: (1595, 68)
Test: (399, 68)
Train_imputed: (1595, 84)
Test_imputed: (399, 84)
